<a href="https://colab.research.google.com/github/ReposofPriyanka/flyrank-ai-internship-ml-track/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract


In [19]:
# Installing the packages needed to query the FlyRank warehouse
!pip -q install duckdb pandas pyarrow

import duckdb
import pandas as pd
from google.colab import userdata

# Get the Hugging Face READ token from Colab Secrets.
# IMPORTANT: Do not paste the actual token into this notebook.
HF_TOKEN = userdata.get("HF_TOKEN")

# Create an in-memory DuckDB connection
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Base location of the FlyRank warehouse
REL = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [20]:
# Inspecting the columns available in the daily performance fact table

schema_df = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    LIMIT 1
""").df()

schema_df

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window


One row represents one content page for one client on one reporting date. I will use a mid-panel month, March 2026, for development and verification rather than the final June 2026 month.

In [21]:
# Verification Query 1 — Grain
# Expected: zero rows.
# If this returns rows, then the stated grain is not unique.

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

**Tables:**

I use fact_content_daily_performance as the main table because it contains daily search and analytics performance for each content page.

**Time window:**

I use March 2026 as the development window. March information is used to create features, while April 2026 is used only as a future outcome. The final June 2026 month is not used to develop the label.

**What I predict / rank:**

My lane is Content Ranking / SEO. I want to score content pages according to signals that can help prioritize pages for refresh or optimization. For this exercise, the future proxy outcome is the percentage change in impressions from March to April.

**Deliberately excluded:**

I deliberately exclude future performance information such as April impressions and future_trend_pct from the features because it would not be known at the March decision moment.

In [22]:
# Verification Query 2 — March row count and date span

march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
""").df()

march_summary

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 3. Verify it with queries (grain, counts, missing values, windows)



| Feature                 | Available when?                                                                                               |
| ----------------------- | ------------------------------------------------------------------------------------------------------------- |
| `impressions_march`     | Knowable at the decision moment because March Search Console impressions have already been observed.          |
| `clicks_march`          | Knowable at the decision moment because March Search Console clicks have already been observed.               |
| `avg_position_march`    | Knowable at the decision moment because March search-position information has already been observed.          |
| `sessions_march`        | Knowable at the decision moment because March GA4 sessions have already been recorded where GA4 is available. |
| `engagement_rate_march` | Knowable at the decision moment because it is calculated from March engaged sessions and sessions.            |


In [23]:
# Verification Query 3 — GA4 availability
# IMPORTANT: Use IS TRUE, not = TRUE.

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS rows_with_ga4_available,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) * 100.0 / COUNT(*) AS pct_available
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,rows_with_ga4_available,pct_available
0,9841378,413966,4.206382


In [24]:
# Build the five-feature frame for March 2026

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature 1: Search impressions
        SUM(gsc_impressions) AS impressions_march,

        -- Feature 2: Search clicks
        SUM(gsc_clicks) AS clicks_march,

        -- Feature 3: Average search position
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,

        -- Feature 4: GA4 sessions
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE NULL
            END
        ) AS sessions_march,

        -- Feature 5: Engagement rate
        100.0 *
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_engaged_sessions
                ELSE NULL
            END
        )
        / NULLIF(
            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_sessions
                    ELSE NULL
                END
            ),
            0
        ) AS engagement_rate_march

    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    WHERE month = '2026-03'

    GROUP BY
        client_hash_id,
        content_hash_id

    ORDER BY
        impressions_march DESC
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,sessions_march,engagement_rate_march
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,8.205128
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,11.029412
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,11.560045
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,12.688172
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,37.0,2.702703


In [25]:
# STEP 5 — Build five features from March 2026

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature 1: Search impressions
        SUM(gsc_impressions) AS impressions_march,

        -- Feature 2: Search clicks
        SUM(gsc_clicks) AS clicks_march,

        -- Feature 3: Average search position
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,

        -- Feature 4: GA4 sessions
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE NULL
            END
        ) AS sessions_march,

        -- Feature 5: GA4 engagement rate
        100.0 *
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_engaged_sessions
                ELSE NULL
            END
        )
        / NULLIF(
            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_sessions
                    ELSE NULL
                END
            ),
            0
        ) AS engagement_rate_march

    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    WHERE month = '2026-03'

    GROUP BY
        client_hash_id,
        content_hash_id

    ORDER BY impressions_march DESC
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,sessions_march,engagement_rate_march
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,8.205128
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,11.029412
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,11.560045
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,12.688172
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,37.0,2.702703


In [26]:
# STEP 6 — Inspect the feature frame

print("Number of rows:", len(feature_frame))
print("Columns:")
print(feature_frame.columns.tolist())

feature_frame.head(10)

Number of rows: 331437
Columns:
['client_hash_id', 'content_hash_id', 'impressions_march', 'clicks_march', 'avg_position_march', 'sessions_march', 'engagement_rate_march']


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,sessions_march,engagement_rate_march
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,8.205128
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,11.029412
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,11.560045
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,12.688172
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,37.0,2.702703
5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.367835,NaN,NaN
6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,4.544203,NaN,NaN
7,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,164.0,9.756098
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,3.186054,NaN,NaN
9,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.766674,2603.0,1.344602


In [27]:
# STEP 7 — Check missing values in the five features

feature_columns = [
    "impressions_march",
    "clicks_march",
    "avg_position_march",
    "sessions_march",
    "engagement_rate_march"
]

missing_values = (
    feature_frame[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

missing_values

,0
engagement_rate_march,72.77
sessions_march,72.70
avg_position_march,47.11
impressions_march,0.00
clicks_march,0.00


In [28]:
# STEP 8 — Create a future outcome using April performance

future_label = con.sql(f"""
    WITH monthly_impressions AS (

        SELECT
            month,
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions

        FROM read_parquet(
            '{REL}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        WHERE month IN ('2026-03', '2026-04')

        GROUP BY
            month,
            client_hash_id,
            content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,

        MAX(
            CASE
                WHEN month = '2026-03'
                THEN impressions
            END
        ) AS march_impressions,

        MAX(
            CASE
                WHEN month = '2026-04'
                THEN impressions
            END
        ) AS april_impressions

    FROM monthly_impressions

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        march_impressions > 0
        AND april_impressions IS NOT NULL
""").df()

future_label.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,april_impressions
0,client_62f4a7e64f5e0096,content_1b1e1380830c6691,2921.0,1059.0
1,client_62f4a7e64f5e0096,content_887859a9bf672686,63.0,118.0
2,client_62f4a7e64f5e0096,content_90e87e327e365568,2071.0,516.0
3,client_62f4a7e64f5e0096,content_fd88056fa55a5921,33.0,14.0
4,client_62f4a7e64f5e0096,content_4eb308d8b5d39926,138.0,62.0


In [29]:
# STEP 9 — Calculate the percentage change in impressions

future_label["future_trend_pct"] = (
    (future_label["april_impressions"] - future_label["march_impressions"])
    / future_label["march_impressions"]
) * 100

future_label.head()

,client_hash_id,content_hash_id,march_impressions,april_impressions,future_trend_pct
0,client_62f4a7e64f5e0096,content_1b1e1380830c6691,2921.0,1059.0,-63.745293
1,client_62f4a7e64f5e0096,content_887859a9bf672686,63.0,118.0,87.301587
2,client_62f4a7e64f5e0096,content_90e87e327e365568,2071.0,516.0,-75.084500
3,client_62f4a7e64f5e0096,content_fd88056fa55a5921,33.0,14.0,-57.575758
4,client_62f4a7e64f5e0096,content_4eb308d8b5d39926,138.0,62.0,-55.072464


In [30]:
# STEP 10 — Combine March features with the future outcome

model_frame = feature_frame.merge(
    future_label[
        [
            "client_hash_id",
            "content_hash_id",
            "future_trend_pct"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print("Rows in final analysis frame:", len(model_frame))

model_frame.head(10)

Rows in final analysis frame: 176737


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,sessions_march,engagement_rate_march,future_trend_pct
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,8.205128,29.529560
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,11.029412,-51.784520
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,11.560045,-29.082476
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,12.688172,37.960779
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,37.0,2.702703,-76.302235
5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.367835,NaN,NaN,-12.609112
6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,4.544203,NaN,NaN,18.086274
7,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,164.0,9.756098,-5.822690
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,3.186054,NaN,NaN,8.937382
9,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.766674,2603.0,1.344602,-22.190987


**Deliberate leakage experiment**

I intentionally created leaky_feature by copying the future outcome future_trend_pct. Because this feature contains the answer that we are trying to predict, it is not available at the March decision moment. The resulting score should therefore become unrealistically close to perfect. I then remove the leaked feature and calculate an honest score using only the five March features.

In [31]:
# STEP 11 — DELIBERATE LEAKAGE EXPERIMENT
# This is intentionally WRONG.
# We are putting the future outcome directly into a feature.

model_frame["leaky_feature"] = model_frame["future_trend_pct"]

model_frame[
    [
        "future_trend_pct",
        "leaky_feature"
    ]
].head()

,future_trend_pct,leaky_feature
0,29.529560,29.529560
1,-51.784520,-51.784520
2,-29.082476,-29.082476
3,37.960779,37.960779
4,-76.302235,-76.302235


In [32]:
# STEP 12 — Show how leakage creates a perfect score

from sklearn.metrics import r2_score

leak_data = model_frame.dropna(
    subset=["future_trend_pct", "leaky_feature"]
)

leaky_r2 = r2_score(
    leak_data["future_trend_pct"],
    leak_data["leaky_feature"]
)

print(f"Leaky R² score: {leaky_r2:.4f}")

Leaky R² score: 1.0000


In [33]:
# STEP 13 — Remove the leaked feature

model_frame = model_frame.drop(
    columns=["leaky_feature"]
)

print("Leaky feature removed.")
print(model_frame.columns.tolist())

Leaky feature removed.
['client_hash_id', 'content_hash_id', 'impressions_march', 'clicks_march', 'avg_position_march', 'sessions_march', 'engagement_rate_march', 'future_trend_pct']


In [34]:
# STEP 14 — Honest quick model using only the five legitimate features

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

feature_columns = [
    "impressions_march",
    "clicks_march",
    "avg_position_march",
    "sessions_march",
    "engagement_rate_march"
]

honest_data = model_frame.dropna(
    subset=["future_trend_pct"]
).copy()

X = honest_data[feature_columns]
y = honest_data["future_trend_pct"]

# Fill missing feature values with the median
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42
)

# Train a simple model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# Evaluate
honest_r2 = r2_score(y_test, predictions)
honest_mae = mean_absolute_error(y_test, predictions)

print(f"Honest R² score: {honest_r2:.4f}")
print(f"Honest MAE: {honest_mae:.4f}")

Honest R² score: -0.3038
Honest MAE: 371.2044


In [35]:
# STEP 15 — Display the final honest feature + outcome frame

final_frame = model_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_march",
        "clicks_march",
        "avg_position_march",
        "sessions_march",
        "engagement_rate_march",
        "future_trend_pct"
    ]
]

final_frame.head(10)

,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,sessions_march,engagement_rate_march,future_trend_pct
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,8.205128,29.529560
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,11.029412,-51.784520
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,11.560045,-29.082476
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,12.688172,37.960779
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,37.0,2.702703,-76.302235
5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.367835,NaN,NaN,-12.609112
6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,4.544203,NaN,NaN,18.086274
7,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,164.0,9.756098,-5.822690
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,3.186054,NaN,NaN,8.937382
9,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.766674,2603.0,1.344602,-22.190987


## 4. Data limits



Limitation: The warehouse is an unbalanced panel, so different clients have different amounts of historical data. GA4 availability also varies across clients and dates. Therefore, some content pages have less complete analytics information than others, and the five-feature frame may not have identical measurement coverage for every page.

In [36]:
# Confirm GA4 availability in the final feature frame

print(
    "Rows with available GA4 sessions:",
    model_frame["sessions_march"].notna().sum()
)

print(
    "Total rows:",
    len(model_frame)
)

Rows with available GA4 sessions: 71888
Total rows: 176737


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.